# 02 — Comparison and Projection

Reframed question (see `DESIGN.md`): across comparable public
evaluations, did measured AI capability decline, increase, or move
differently by domain, from **late 2025 to mid-2026**?

The original September 2025 vs September 2026 headline is not
supported by the data: METR Time Horizon 1.1 reaches April 2026
model releases, LiveBench v1 reaches June 2026 and skips September
2025, and SWE-bench dates are submissions, not model releases.

No single merged intelligence score is computed anywhere in this
notebook. Each source is reported separately, in its own units.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import comparison
print("comparison module loaded from", ROOT / "src" / "comparison.py")


## 1. METR — autonomous task horizon

Source: `https://metr.org/assets/benchmark_results_1_1.yaml`
Unit: one model/agent fit, Time Horizon 1.1 only.
Date: `release_date` only. Cohorts: before 2025-09-01 vs
2025-09-01 to 2026-04-30. Sensitivity: 2025-01-01 to 2025-08-31.


In [ ]:
import urllib.request, yaml
metr_url = "https://metr.org/assets/benchmark_results_1_1.yaml"
with urllib.request.urlopen(metr_url, timeout=90) as r:
    doc = yaml.safe_load(r.read())
rows = []
for model, item in doc["results"].items():
    p = item.get("metrics", {}).get("p50_horizon_length", {})
    rows.append({"model": model, "release_date": item.get("release_date"),
                 "methodology": item.get("benchmark_name"),
                 "p50_minutes": p.get("estimate")})
result = comparison.metr_cohort_split(rows)
print(result)
print("METR doubling time (from 2023 on):", doc.get("doubling_time_in_days"))
# Observed on 2026-11-21: v1.1 n=23; A n=14 median 20.4 min;
# B n=9 median 349.5 min; recent-A n=5 median 100.5 min.


## 2. SWE-bench Verified — coding systems

Source: SWE-bench leaderboards JSON (Verified board).
Unit: one submission. Date: `submission_date` only.
Cohorts: Sep–Dec 2025 vs Jan–Sep 2026, first submission per
(system, model) per window, with a max-resolved sensitivity check.


In [ ]:
import json, urllib.request
swe_url = "https://raw.githubusercontent.com/SWE-bench/swe-bench.github.io/master/data/leaderboards.json"
with urllib.request.urlopen(swe_url, timeout=90) as r:
    data = json.load(r)
verified = next(b["results"] for b in data["leaderboards"] if b["name"] == "Verified")
rows = [{"system": x.get("name"), "model": x.get("model_display"),
         "submission_date": x.get("date"), "resolved_pct": x.get("resolved")}
        for x in verified]
for window in ("A", "B"):
    print(window, comparison.swebench_cohort(rows, window))
    print(window, "max sensitivity:", comparison.swebench_max_sensitivity(rows, window))
# Observed on 2026-11-21: A raw n=32, dedup n=31, median 71.2%;
# B raw n=13, dedup n=13, median 72.8% — B is February 2026 only,
# no submissions after 2026-02-26 in the file.


## 3. LiveBench v1 — not verifiable on 2026-11-21

The paired Nov 2025 vs Jun 2026 comparison in `DESIGN.md` could
not be run from its original source on 2026-11-21:
`github.com/LiveBench/new-livebench` returned 404, and
`livebench.ai/table_2025_11_25.csv` / `table_2026_06_25.csv`
returned 503 / 404. The code below attempts the fetch and reports
the failure instead of inventing numbers. `comparison.livebench_paired_diff`
is implemented and hand-tested, and is ready to run when the tables
are fetchable again.

LiveBench 2.0, first released 2026-10-14, is a different benchmark
with different tasks and scoring, so its scores are **not**
comparable with v1. Its 2026-10-25 snapshot is reported separately
below, as a single snapshot, not a trend.


In [ ]:
import urllib.request, urllib.error
for release in ("2025_11_25", "2026_06_25"):
    url = f"https://livebench.ai/table_{release}.csv"
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            print(release, "HTTP", r.status, "bytes", len(r.read()))
    except Exception as e:
        print(release, "NOT FETCHABLE on 2026-11-21:", type(e).__name__, e)


In [ ]:
import pandas as pd
# LiveBench 2.0 snapshot only — not comparable with v1.
lb2_url = "https://livebench.ai/data/20261025/table.csv"
lb2 = pd.read_csv(lb2_url)
print("LiveBench 2.0, 2026-10-25, n =", len(lb2))
print(lb2.median(numeric_only=True))
# Observed on 2026-11-21: n=32; medians — long_agentic 29.4,
# reasoning 86.7, coding 85.1, browser 59.9, data_analysis 76.6,
# research 77.3, design 77.1.


## 4. Projection

See `PROJECTION.md` for the full projection, with assumptions and
why it could be wrong. The calculation below is the METR part only:
start from the observed B median (349.5 minutes, early 2026) and
apply METR's from-2023 doubling time of 128.7 days.


In [ ]:
import math
from datetime import date, timedelta
start_median = 349.530732
doubling_days = 128.744
start = date(2026, 4, 7)  # last model release in the METR file
for months, label in [(4, "Aug 2026"), (8, "Dec 2026-Jan 2027"), (12, "Apr-May 2027")]:
    days = months * 30
    projected = start_median * (2 ** (days / doubling_days))
    print(label, f"~{projected:.0f} minutes (~{projected/60:.1f} hours)")
print("Assumption: doubling continues at ~129 days, methodology stable.")
print("This is extrapolation, not observation — see PROJECTION.md.")


## 5. Synthesis (per source only)

| Source | Observed | Finding |
|---|---|---|
| METR v1.1 | A n=14 median 20.4 min; B n=9 median 349.5 min; recent-A n=5 median 100.5 min | Frontier horizon rose sharply, small n |
| SWE-bench Verified | A n=31 median 71.2%; B n=13 median 72.8% (Feb 2026 only) | Essentially flat / plateau |
| LiveBench v1 | Not verifiable 2026-11-21 | No paired finding can be claimed |
| LiveBench 2.0 | 2026-10-25 snapshot, n=32, long-agentic median 29.4 | Single snapshot, not a trend |

Human preference / everyday product reliability is out of scope:
benchmark scores alone cannot show whether a product feels more
reliable. That needs repeated trials on fixed tasks with stable
prompts, settings, tools, and success criteria.
